# Revised Visualization Notebook

This notebook follows the feedback in `Visual fix(2).docx`:

1. Add statistics indicators such as mean, median, variance and standard deviation
2. Use a pie chart for positive vs negative class distribution
3. Split the bar charts into `LR + ML models` and `LR + embedding models`
4. Use Accuracy, Precision, Recall and F1 for the classic-model chart
5. Keep a clear note that embedding Precision / Recall / confusion matrices need to be exported by the teammate if not saved locally


In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.tree import DecisionTreeClassifier
from wordcloud import WordCloud

sns.set_theme(style='whitegrid')
PURPLE_BG = '#3A295B'
CYAN = '#65F0F9'
BLUE = '#7BA5D8'
DEEP_PURPLE = '#4B2A8C'
LIGHT_PURPLE = '#A888D5'
OUTLINE = '#8B5CF6'
plt.rcParams['figure.dpi'] = 150
plt.rcParams['axes.facecolor'] = '#F8F5FF'
plt.rcParams['figure.facecolor'] = 'white'

ROOT = Path.cwd()

def locate_file(name: str) -> Path:
    for candidate in [ROOT / name, ROOT.parent / name, ROOT.parent / '527_project' / name]:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f'Cannot find {name}')

train_path = locate_file('train_fixed_split.csv')
test_path = locate_file('test_fixed_split.csv')

train_df = pd.read_csv(train_path).dropna(subset=['text']).copy()
test_df = pd.read_csv(test_path).dropna(subset=['text']).copy()
full_df = pd.concat([train_df, test_df], ignore_index=True)
full_df['word_count'] = full_df['text'].astype(str).str.split().str.len()
full_df.head()


In [ ]:
stats_df = pd.DataFrame({
    'Statistic': ['Mean', 'Median', 'Variance', 'Std'],
    'Word Count': [
        full_df['word_count'].mean(),
        full_df['word_count'].median(),
        full_df['word_count'].var(),
        full_df['word_count'].std(),
    ]
})
stats_df['Word Count'] = stats_df['Word Count'].round(2)
stats_df


In [ ]:
label_counts = full_df['label'].value_counts().sort_index()
pie_labels = ['Negative', 'Positive']
pie_values = [label_counts.get(0, 0), label_counts.get(1, 0)]
pie_colors = [LIGHT_PURPLE, CYAN]

fig, ax = plt.subplots(figsize=(6, 6))
wedges, texts, autotexts = ax.pie(
    pie_values,
    labels=pie_labels,
    autopct='%1.1f%%',
    startangle=90,
    colors=pie_colors,
    wedgeprops={'width': 0.52, 'edgecolor': 'white'}
)
ax.set_title('Positive vs Negative Distribution', fontsize=16, weight='bold', color=DEEP_PURPLE)
plt.show()


In [ ]:
vectorizer = TfidfVectorizer(max_features=10000, min_df=2, ngram_range=(1, 2))
X_train = vectorizer.fit_transform(train_df['text'])
X_test = vectorizer.transform(test_df['text'])
y_train = train_df['label']
y_test = test_df['label']

classic_models = {
    'Logistic Regression (Baseline)': LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced'),
    'Decision Tree': DecisionTreeClassifier(max_depth=10, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=50, learning_rate=0.1, random_state=42),
}

classic_results = []
conf_mats = {}

for name, model in classic_models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    classic_results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred, zero_division=0),
        'Recall': recall_score(y_test, pred, zero_division=0),
        'F1': f1_score(y_test, pred, zero_division=0),
    })
    conf_mats[name] = confusion_matrix(y_test, pred)

classic_df = pd.DataFrame(classic_results)
classic_df


In [ ]:
classic_plot = classic_df.melt(id_vars='Model', value_vars=['Accuracy', 'Precision', 'Recall', 'F1'], var_name='Metric', value_name='Score')
metric_palette = {
    'Accuracy': CYAN,
    'Precision': BLUE,
    'Recall': DEEP_PURPLE,
    'F1': LIGHT_PURPLE,
}

fig, ax = plt.subplots(figsize=(11, 5.8))
sns.barplot(data=classic_plot, x='Model', y='Score', hue='Metric', palette=metric_palette, ax=ax)
ax.set_title('LR + Machine Learning Models', fontsize=16, weight='bold', color=DEEP_PURPLE)
ax.set_xlabel('')
ax.set_ylabel('Score', color=DEEP_PURPLE)
ax.set_ylim(0, 1)
ax.tick_params(axis='x', rotation=14, colors=DEEP_PURPLE)
ax.tick_params(axis='y', colors=DEEP_PURPLE)
ax.legend(frameon=True, facecolor='white', edgecolor=LIGHT_PURPLE)
for container in ax.containers:
    ax.bar_label(container, fmt='%.4f', padding=3, fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.flatten()

for ax, (name, matrix) in zip(axes, conf_mats.items()):
    sns.heatmap(
        matrix,
        annot=True,
        fmt='d',
        cmap=sns.blend_palette([LIGHT_PURPLE, DEEP_PURPLE, CYAN], as_cmap=True),
        cbar=False,
        xticklabels=['Pred Neg', 'Pred Pos'],
        yticklabels=['True Neg', 'True Pos'],
        ax=ax,
    )
    ax.set_title(name, fontsize=12, weight='bold', color=DEEP_PURPLE)
    ax.tick_params(colors=DEEP_PURPLE)

plt.suptitle('Confusion Matrix Comparison for LR + ML Models', fontsize=16, weight='bold', y=1.02, color=DEEP_PURPLE)
plt.tight_layout()
plt.show()


In [ ]:
embedding_df = pd.DataFrame([
    {'Model': 'Logistic Regression (Baseline)', 'Accuracy': 0.9262, 'F1': 0.9606, 'Precision': 0.9437, 'Recall': 0.9781, 'AUC': None},
    {'Model': 'Word2Vec', 'Accuracy': 0.9195, 'F1': 0.9039, 'Precision': None, 'Recall': None, 'AUC': 0.7322},
    {'Model': 'GloVe', 'Accuracy': 0.9060, 'F1': 0.8914, 'Precision': None, 'Recall': None, 'AUC': 0.7927},
    {'Model': 'BERT', 'Accuracy': 0.8926, 'F1': 0.8881, 'Precision': None, 'Recall': None, 'AUC': 0.7310},
])
embedding_df


In [ ]:
embedding_plot = embedding_df.melt(id_vars='Model', value_vars=['Accuracy', 'F1', 'AUC'], var_name='Metric', value_name='Score')
embedding_plot = embedding_plot.dropna()
embedding_palette = {'Accuracy': CYAN, 'F1': BLUE, 'AUC': LIGHT_PURPLE}

fig, ax = plt.subplots(figsize=(10, 5.6))
sns.barplot(data=embedding_plot, x='Model', y='Score', hue='Metric', palette=embedding_palette, ax=ax)
ax.set_title('LR + Embedding Models', fontsize=16, weight='bold', color=DEEP_PURPLE)
ax.set_xlabel('')
ax.set_ylabel('Score', color=DEEP_PURPLE)
ax.set_ylim(0, 1)
ax.tick_params(axis='x', rotation=12, colors=DEEP_PURPLE)
ax.tick_params(axis='y', colors=DEEP_PURPLE)
ax.legend(frameon=True, facecolor='white', edgecolor=LIGHT_PURPLE)
for container in ax.containers:
    ax.bar_label(container, fmt='%.4f', padding=3, fontsize=8)
plt.tight_layout()
plt.show()

print('Note: Word2Vec / GloVe / BERT precision, recall and confusion matrices are not available in the saved local outputs. Ask the teammate who ran word_embedding.ipynb to export the full classification reports if the final PPT requires them.')


In [ ]:
custom_stopwords = set(ENGLISH_STOP_WORDS) | {
    'album', 'albums', 'dark', 'floyd', 'good', 'great', 'just', 'like', 'moon',
    'movie', 'movies', 'music', 'netflix', 'one', 'pink', 'que', 'really', 'series',
    'show', 'shows', 'song', 'songs', 'story', 'time', 'watch', 'watched', 'watching'
}

def prepare_text(series):
    tokens = []
    for text in series.astype(str):
        tokens.extend([w for w in re.findall(r'[a-zA-Z]{3,}', text.lower()) if w not in custom_stopwords])
    return ' '.join(tokens)

positive_text = prepare_text(train_df.loc[train_df['label'] == 1, 'text'])
negative_text = prepare_text(train_df.loc[train_df['label'] == 0, 'text'])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
configs = [
    ('Positive Reviews', positive_text, CYAN),
    ('Negative Reviews', negative_text, LIGHT_PURPLE),
]

for ax, (title, text, color) in zip(axes, configs):
    wc = WordCloud(width=900, height=500, background_color=PURPLE_BG, colormap='cool', collocations=False).generate(text)
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(title, fontsize=15, weight='bold', color=color)
    ax.axis('off')

plt.tight_layout()
plt.show()
